In [80]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import torchaudio 
import torchaudio.transforms as T

from datasets import load_dataset

import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from corpus.single_word import SingleWordDataset as Dataset


In [7]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words']  # Name of data splits to be used as validation set


In [8]:
dataset = Dataset(path, dev_split, None, 1)


Using custom data configuration default-d77d8d0e2c068ac8
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-d77d8d0e2c068ac8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)


In [9]:
sampling_rate = 16000

In [12]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

Downloading:   0%|          | 0.00/360M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Check performance on demo dataset

In [61]:
dataset2 = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation")
dataset2 = dataset2.sort("id")
sampling_rate = dataset2.features["audio"].sampling_rate

Downloading:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Downloading:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

0 examples [00:00, ? examples/s]

Dataset librispeech_asr downloaded and prepared to /home/imgriff/.cache/huggingface/datasets/hf-internal-testing___librispeech_asr/clean/2.1.0/d3bc4c2bc2078fcde3ad0f0f635862e4c0fef78ba94c4a34c4c250a097af240b. Subsequent calls will reuse this data.


In [62]:
dataset2

Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 73
})

In [69]:
# results = []

model = model.cuda()
for ix in range(5):
    true = dataset2[ix]["text"]
    inputs = processor(dataset2[ix]["audio"]['array'], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
    print(f'guess: {transcription[0]}')
    print(f'truth: {true}')
    print()
    
#     results.append({'hyp': transcription[0],
#                     'truth': true})

guess: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
truth: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL

guess: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
truth: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER

guess: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMANUS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND
truth: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND

guess: HE HAS GRAVED DOUBTS WHETHER SIR FREDERICK LAYTON'S WORK IS READY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA
truth: HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LEIGHTON'S WORK IS REALLY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA

guess: LINILL'S PI

We can see this is a character model based on the handful of errors made. Overall performance looks good.

## Check on Single Word Corpora

no language model is getting used, so things may be messy 

In [70]:
dataset.dataset

Dataset({
    features: ['Unnamed: 0', 'word', 'wav_path'],
    num_rows: 200
})

In [81]:
resampler = T.Resample(44100, 16000, lowpass_filter_width=64,
                   rolloff=0.9475937167399596, 
                   resampling_method="kaiser_window",beta=14.769656459379492, dtype=torch.float32)

def load_audio(example):
    wav, sr = torchaudio.load(example['wav_path'])
    wav = resampler(wav)
    example['audio'] = wav
    example['sampling_rate'] = 16000
    return example


In [82]:
updated_dataset = dataset.dataset.map(load_audio)

0ex [00:00, ?ex/s]

In [83]:
updated_dataset[0]

{'Unnamed: 0': 0,
 'word': 'THE',
 'wav_path': '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/The.wav',
 'audio': [[-1.0360400665376801e-05,
   -3.228379864594899e-05,
   -2.9783692298224196e-05,
   -3.093810300924815e-05,
   -3.0126482670311816e-05,
   -3.112291233264841e-05,
   -2.9290924430824816e-05,
   -5.418122236733325e-05,
   -6.27073532086797e-05,
   -6.0693666455335915e-05,
   -5.9715188399422914e-05,
   -7.302222365979105e-05,
   -9.539120947010815e-05,
   -8.93403121153824e-05,
   -9.276704804506153e-05,
   -0.00011355966125847772,
   -0.00012602814240381122,
   -0.00011839936632895842,
   -0.00012942183820996433,
   -0.00015273068856913596,
   -0.00015162889030762017,
   -0.00017005435074679554,
   -0.00018827164603862911,
   -0.00017799391935113817,
   -0.00020856880291830748,
   -0.0002118796546710655,
   -0.00022120244102552533,
   -0.00024273886810988188,
   -0.0002550320641603321,
   -0.00027793567278422415,
   -0.0002726104576140642,
   

In [84]:
results = []

model = model.cuda()
for sample in updated_dataset:
    true = sample["word"]
    inputs = processor(sample["audio"], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
    print(f'guess: {transcription[0]}')
    print(f'truth: {true}')
    print()
    
    results.append({'hyp': transcription[0],
                    'truth': true})

guess: THE
truth: THE

guess: AND
truth: AND

guess: TWO
truth: TO

guess: OF
truth: OF

guess: A
truth: A

guess: THAT
truth: THAT

guess: IN
truth: IN

guess: I
truth: I

guess: YOU
truth: YOU

guess: IS
truth: IS

guess: IT
truth: IT

guess: WAS
truth: WAS

guess: FOUR
truth: FOR

guess: SO
truth: SO

guess: THIS
truth: THIS

guess: WE
truth: WE

guess: BUT
truth: BUT

guess: WITH
truth: WITH

guess: ON
truth: ON

guess: HAVE
truth: HAVE

guess: HE
truth: HE

guess: AS
truth: AS

guess: B
truth: BE

guess: R
truth: ARE

guess: THEY
truth: THEY

guess: NOT
truth: NOT

guess: AT
truth: AT

guess: LIKE
truth: LIKE

guess: WHAT
truth: WHAT

guess: NO
truth: KNOW

guess: HIS
truth: HIS

guess: ALL
truth: ALL

guess: ABOUT
truth: ABOUT

guess: ONE
truth: ONE

guess: MORE
truth: OR

guess: HAD
truth: HAD

guess: IF
truth: IF

guess: MY
truth: MY

guess: DE
truth: DO

guess: FROM
truth: FROM

guess: THERE
truth: THERE

guess: JUST
truth: JUST

guess: CAN
truth: CAN

guess: YOUR
truth: YOUR


tensor([[-0.0008, -0.0046, -0.0041,  ...,  0.0008,  0.0008,  0.0008]],
       device='cuda:0')

In [85]:
import pandas as pd

In [86]:
results = pd.DataFrame(results)

In [87]:
results.head(50)

,hyp,truth
0,THE,THE
1,AND,AND
2,TWO,TO
3,OF,OF
4,A,A
5,THAT,THAT
6,IN,IN
7,I,I
8,YOU,YOU
9,IS,IS


In [92]:
top_match_guesses = results[(results['hyp'] == results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses

173 matched guesses; 86.5% correct


,hyp,truth
0,THE,THE
1,AND,AND
3,OF,OF
4,A,A
5,THAT,THAT
...,...,...
193,WHILE,WHILE
194,BIG,BIG
195,TWENTY,TWENTY
196,EACH,EACH
